In [5]:
## View ESP of molecular moment
fchk_file='run_psi4/SP_PBE0_aug-cc-pvtz_h2o.fchk'
mol_mom='run_psi4/molmom.mom'
dis_mom='horton_run/moment.mom'

import os
from mayavi_molecule_viewer.main import plot_esp_surface
from python_scripts.run_process import run_subrocess

from python_scripts.conda_env import get_python_for_script
from qcp_global_utils.environment.file_handling import pdtc_file
from pydantic import validate_call; val_call=validate_call(config=dict(arbitrary_types_allowed=True, validate_return=True))

from contextlib import contextmanager
import time

In [6]:

@contextmanager
def run_info(tag):
    start=time.time()
    print(f"#-- Starting {tag}")
    try:
        yield
    finally:
        print(f"--- Finished after {time.time()-start:.2f} [s]")


@val_call
def make_surface(fchk_file:pdtc_file, grid_spacing:float=0.5, iso_density: float=0.001) -> pdtc_file:
    """ Returns file name of surface 
    
    """
    python,script=get_python_for_script('qcp_dens','calc_IsoDensSurf.py')

    import subprocess as sp
    cmd=f"{python} {script} {os.path.realpath(fchk_file)} --grid_spacing {grid_spacing} --iso_dens {iso_density}"
    run_subrocess(cmd, 'make_surface')
    return os.path.realpath('make_surface/dummy_surface.json')

@val_call
def make_densesp_map(fchk_file, surface_file) -> pdtc_file:
    python,script=get_python_for_script('qcp_dens','calc_esp.py')
    run_dir,ofile="make_dens_esp","dens_esp.json"
    cmd=f"{python} {script} --dens {os.path.realpath(fchk_file)} --surf {os.path.realpath(surface_file)} --engine numpy -o {ofile}"
    run_subrocess(cmd, run_dir)
    return os.path.realpath(os.path.join(run_dir,ofile))

@val_call
def make_mul_map(moment_file:pdtc_file, surface_file:pdtc_file=None, run_dir='make_mom_esp') -> pdtc_file:
    python,script=get_python_for_script('qcp_dens','calc_mul_esp.py')
    ofile='mom_esp.json'
    cmd=f"{python} {script} --mom {os.path.realpath(moment_file)} --grid {os.path.realpath(surface_file)}  --ranks \'max4\' -o {ofile}"
    run_subrocess(cmd, run_dir)
    return os.path.realpath(os.path.join(run_dir,ofile))


# make context window with info and timing
with run_info('run_surface'):
    surface_file=make_surface(fchk_file)
with run_info('run_surface'):
    esp_map=make_densesp_map(fchk_file, surface_file)
with run_info('run_surface'):
    molmom_map=make_mul_map(mol_mom,surface_file, run_dir='make_molmom_esp')
with run_info('run_surface'):
    dismom_map=make_mul_map(dis_mom,surface_file, run_dir='make_dismom_esp')

#-- Starting run_surface
Started process with PID 84095
--- Finished after 0.48 [s]
#-- Starting run_surface
Started process with PID 84115
--- Finished after 6.35 [s]
#-- Starting run_surface
Started process with PID 84147
--- Finished after 0.28 [s]
#-- Starting run_surface
Started process with PID 84150
--- Finished after 0.30 [s]


In [ ]:
# So first 
from mayavi import mlab
import os
mlab.init_notebook(backend='ipy') # Allows for interactive matlab plots
view_object=plot_esp_surface(esp=dismom_map,esp_ref=esp_map, geom=fchk_file, 
                            #opacity=0.75, see_through=one_side_see_through
)
view_object[0]

Notebook initialized with ipy backend.


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\xf4\x00\x00\x01\xf4\x08\x02\x00\x00\x00D\xb4H\xd…

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /home/bruno/0_Software/miniconda3/envs/alston_run_horton_mayavi/lib/python3.15/site-packages/ipy │
│ widgets/widgets/widget.py:191 in __call__                                                        │
│                                                                                                  │
│   190 │   │   │   try:                                                                           │
│ ❱ 191 │   │   │   │   local_value = callback(*args, **kwargs)                                    │
│   192 │   │   │   except Exception as e:                                                         │
│                                                                                                  │
│ /home/bruno/0_Software/miniconda3/envs/alston_run_horton_mayavi/lib/python3.15/site-packages/may │
│ avi/tools/remote/ipy_remote.py:87 in handle_ipyevent                                             │
│                                                                                                  │
│    86 │   │   │   key_sym = key                                                                  │
│ ❱  87 │   │   │   self.on_key_press(ctrl, shift, key, key_sym)                                   │
│    88 │   │   elif type == 'keyup':                                                              │
│                                                                                                  │
│ /home/bruno/0_Software/miniconda3/envs/alston_run_horton_mayavi/lib/python3.15/site-packages/may │
│ avi/tools/remote/remote_widget.py:132 in on_key_press                                            │
│                                                                                                  │
│   131 │   │   """                                                                                │
│ ❱ 132 │   │   self.send('SetEventInformationFlipY', self._saveX, self._saveY,                    │
│   133 │   │   │   │     ctrl, shift, key, 0, key_sym)                                            │
│                                                                                                  │
│ /home/bruno/0_Software/miniconda3/envs/alston_run_horton_mayavi/lib/python3.15/site-packages/may │
│ avi/tools/remote/remote_widget.py:44 in send                                                     │
│                                                                                                  │
│    43 │   def send(self, method, *args):                                                         │
│ ❱  44 │   │   self.scene_proxy.call_rwi(*((method, ) + args))                                    │
│    45                                                                                            │
│                                                                                                  │
│ /home/bruno/0_Software/miniconda3/envs/alston_run_horton_mayavi/lib/python3.15/site-packages/may │
│ avi/tools/remote/remote_scene.py:153 in call_rwi                                                 │
│                                                                                                  │
│   152 │   │                                                                                      │
│ ❱ 153 │   │   getattr(self.rwi, method)(*args)                                                   │
│   154                                                                                            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValueError: SetEventInformationFlipY argument 5: